#### Provision an Azure AI Language resource

- Open the Azure portal at https://portal.azure.com, and sign in using the Microsoft account associated with your Azure subscription.
- Select Create a resource.
- In the search field, search for Language service. Then, in the results, select Create under Language Service.
- Select Continue to create your resource.
- Provision the resource using the following settings:
    - Subscription: Your Azure subscription.
    - Resource group: Choose or create a resource group.
R   - egion:Choose any available region
    - Name: Enter a unique name.
    - Pricing tier: Select F0 (free), or S (standard) if F is not available.
    - Responsible AI Notice: Agree.
- Select Review + create, then select Create to provision the resource.
- Wait for deployment to complete, and then go to the deployed resource.
- View the Keys and Endpoint page in the Resource Management section. You will need the information on this page later in the exercise.

⚙️ Install the following package

- >poetry add azure-ai-textanalytics
- >poetry add dotenv

In [2]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

from dotenv import load_dotenv
import os

In [5]:
## Load the environment variables
load_dotenv()
ai_endpoint = os.getenv('AI_SERVICE_ENDPOINT')
ai_key = os.getenv('AI_SERVICE_KEY')

In [3]:
# Source of the review comments. 
# 🌍 https://github.com/MicrosoftLearning/mslearn-ai-language/tree/main/Labfiles/01-analyze-text/Python/text-analysis/reviews
review_comments="""
Tired hotel with poor service
The Royal Hotel, London, United Kingdom
5/6/2018
This is a old hotel (has been around since 1950's) and the room furnishings are average - becoming a bit old now and require changing. The internet didn't work and had to come to one of their office rooms to check in for my flight home. The website says it's close to the British Museum, but it's too far to walk.
"""

In [6]:
credential = AzureKeyCredential(key=ai_key)

# create the language client
client = TextAnalyticsClient(endpoint=ai_endpoint, credential=credential)

🗣️ Detect language.
- detect_language except a list of string

In [11]:
detect_language = client.detect_language([review_comments])

In [17]:
print(detect_language[0].primary_language.name)
print(detect_language[0].primary_language.confidence_score)

English
1.0


In [18]:
sentiment_obj = client.analyze_sentiment([review_comments])

### 😃 analyze_sentiment method returns
- Overall sentiment 
- Sentence wise sentiment

In [ ]:
# Overall Sentiments
overall_sentiment = sentiment_obj[0].sentiment
print(f"Overall sentiment: {overall_sentiment}")
overall_confidence_score = sentiment_obj[0].confidence_scores
print(f"Overall sentiment: {overall_confidence_score}")

Overall sentiment: negative
Overall sentiment: {'positive': 0.0, 'neutral': 0.0, 'negative': 1.0}


In [29]:
for sentence_result in sentiment_obj[0].sentences:
    print(f"SENTIMENT: {sentence_result.sentiment}")
    print(f"Confidence Score: {sentence_result.confidence_scores}")
    print(f"TEXT: {sentence_result.text}")
    print(f"😀")


SENTIMENT: negative
Confidence Score: {'positive': 0.0, 'neutral': 0.0, 'negative': 1.0}
TEXT:  Tired hotel with poor service The Royal Hotel, London, United Kingdom 5/6/2018 This is a old hotel (has been around since 1950's) and the room furnishings are average - becoming a bit old now and require changing. 
😀
SENTIMENT: negative
Confidence Score: {'positive': 0.0, 'neutral': 0.0, 'negative': 1.0}
TEXT: The internet didn't work and had to come to one of their office rooms to check in for my flight home. 
😀
SENTIMENT: neutral
Confidence Score: {'positive': 0.0, 'neutral': 0.79, 'negative': 0.21}
TEXT: The website says it's close to the British Museum, but it's too far to walk. 
😀


### Extract Key Phrases

In [30]:
key_phrases = client.extract_key_phrases([review_comments])

In [34]:
for key_ph in key_phrases[0].key_phrases:
    print(key_ph)

The Royal Hotel
Tired hotel
old hotel
poor service
United Kingdom
room furnishings
office rooms
flight home
British Museum
London
changing
internet
website
1950


### 🐧Extract entities

In [37]:
entities = client.recognize_entities(documents=[review_comments])

In [39]:
for entity in entities[0].entities:
    print(entity)


{'text': 'hotel', 'category': 'Location', 'subcategory': None, 'length': 5, 'offset': 7, 'confidence_score': 0.94}
{'text': 'Hotel', 'category': 'Location', 'subcategory': None, 'length': 5, 'offset': 41, 'confidence_score': 0.53}
{'text': 'London, United Kingdom', 'category': 'Address', 'subcategory': None, 'length': 22, 'offset': 48, 'confidence_score': 0.84}
{'text': '5/6/2018', 'category': 'DateTime', 'subcategory': 'Date', 'length': 8, 'offset': 71, 'confidence_score': 1.0}
{'text': 'hotel', 'category': 'Location', 'subcategory': None, 'length': 5, 'offset': 94, 'confidence_score': 0.83}
{'text': 'since 1950', 'category': 'DateTime', 'subcategory': 'DateRange', 'length': 10, 'offset': 117, 'confidence_score': 0.99}
{'text': 'now', 'category': 'DateTime', 'subcategory': None, 'length': 3, 'offset': 189, 'confidence_score': 1.0}
{'text': 'one', 'category': 'Quantity', 'subcategory': 'Number', 'length': 3, 'offset': 259, 'confidence_score': 0.8}
{'text': 'office rooms', 'category': '

### 🔗Extract linked entities

In [ ]:
text_for_analysis = """
I saw Venus shining in the sky.
"""

linked_entities = client.recognize_linked_entities(documents=[text_for_analysis])

In [49]:
for entity in linked_entities[0].entities:
    print(entity.name)
    print(entity.url)

Venus
https://en.wikipedia.org/wiki/Venus
